# VisProbe Walkthrough

End-to-end compositional robustness test: environmental degradation x adversarial attack across a severity sweep, with automatic checkpoint + resume and per-model GPU swapping. We then save the results and reload them for offline inspection.

Runs on CPU (slowly) or GPU. We use a tiny random model + random inputs to keep the notebook self-contained; swap in `torchvision.models.resnet50(pretrained=True)` and real data for a real run.

## 1. Setup

In [ ]:
import torch
import torch.nn as nn

from visprobe import (
    CompositionalExperiment,
    CompositionalResults,
    get_standard_perturbations,
)

torch.manual_seed(0)


class TinyClassifier(nn.Module):
    def __init__(self, n_classes: int = 10):
        super().__init__()
        self.conv = nn.Conv2d(3, 16, 3, padding=1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(16, n_classes)

    def forward(self, x):
        x = torch.relu(self.conv(x))
        x = self.pool(x).view(x.size(0), -1)
        return self.fc(x)


models = {"tiny": TinyClassifier()}
images = torch.rand(32, 3, 32, 32)
labels = torch.randint(0, 10, (32,))

## 2. Configure and run the experiment

We sweep severity in [0, 1] over the 4 standard perturbations (blur, noise, brightness, lowlight). Adversarial epsilon scales linearly with severity. The run uses APGD-CE; switch `attack` to `"autoattack-standard"` for the full AutoAttack suite (~5x slower) or `"none"` to test environmental-only robustness.

In [ ]:
experiment = CompositionalExperiment(
    models=models,
    images=images,
    labels=labels,
    env_strategies=get_standard_perturbations(),
    attack="none",  # set to 'autoattack-apgd-ce' for real adversarial eval
    severities=[0.0, 0.25, 0.5, 0.75, 1.0],
    eps_fn=lambda s: (8 / 255) * s,
    checkpoint_dir="./checkpoints/demo",
    batch_size=16,
    device="cuda" if torch.cuda.is_available() else "cpu",
    verbose=True,
)

results = experiment.run()  # auto-resumes if ./checkpoints/demo already has progress

## 3. Inspect and visualize

`print_summary` gives accuracy ranges and AUC per (model, scenario). `plot_compositional` renders one subplot per scenario, with one line per model.

In [ ]:
results.print_summary()
results.plot_compositional()

## 4. Save and reload

`save` writes a directory containing the pickled results, a JSON summary, and metadata. `load` is a classmethod that needs no GPU and no models -- useful for analysis on a laptop after the run finishes on a server.

In [ ]:
results.save("./results/demo")

reloaded = CompositionalResults.load("./results/demo")
reloaded.print_summary()

# Drill into a specific cell:
cell = reloaded.get_result("tiny", "blur", 0.5)
print(f"\nblur @ severity=0.5: accuracy={cell.accuracy:.3f}, n_samples={cell.n_samples}")

experiment.cleanup()